# Day 13 · Raw NOAA data validation

This notebook validates the raw NOAA IBTrACS and OISST inputs and writes both processed track CSVs. [Data workflow](DATA_WORKFLOW.md) lists the source downloads, input bounds and dependencies. The notebook requires a Python 3 kernel, R with `readr` and `dplyr`, and the NetCDF utility `ncdump`.

## 1. Configure source paths

The default locations are `raw/ibtracs.last3years.list.v04r01.csv` and `raw/oisst/`, relative to the project root. The input paths and output directory may be configured through the variables below.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

ibtracs_csv = Path(os.environ.get("IBTRACS_CSV", "raw/ibtracs.last3years.list.v04r01.csv"))
oisst_dir = Path(os.environ.get("OISST_DIR", "raw/oisst"))
output_dir = Path(os.environ.get("OUT_DIR", "generated"))
oisst_files = sorted(oisst_dir.glob("ncdcOisst21NrtAgg_*.nc"))
oisst_files += sorted(oisst_dir.glob("ncdcOisst21Agg_*.nc"))

if not ibtracs_csv.is_file():
    raise FileNotFoundError(ibtracs_csv)
if not oisst_files:
    raise FileNotFoundError(f"No OISST NetCDF files found in {oisst_dir}")

## 2. Validate and process the raw sources

- `01_prepare_tracks.R` checks the IBTrACS columns and parsing, then checks storm IDs, UTC timestamps, coordinate ranges and non-negative winds for Western Pacific observations in 2024–2026. Missing positions or winds are reported and excluded by the map filters. Retained observation keys must be unique.
- `02_match_oisst.R` checks each NetCDF file for Celsius units, the expected 0.25° grid axes, valid and non-overlapping UTC dates, array dimensions and the allowed SST range. It then matches each retained track point to the nearest grid cell for its UTC date and records masked cells and missing dates separately.

The workflow writes `typhoon_tracks_2024_2026.csv` and `typhoon_tracks_oisst_2024_2026.csv` to the configured output directory.

In [ ]:
result = subprocess.run(
    [
        sys.executable, "03_validate_typhoon_data.py",
        "--ibtracs", str(ibtracs_csv),
        "--oisst", *map(str, oisst_files),
        "--output-dir", str(output_dir),
    ],
    check=True,
    stdout=subprocess.PIPE,
    text=True,
)
print(result.stdout, end="")